# CNN Baseline para Clasificación de Neumonía en Radiografías de Tórax

## Bienvenida al proyecto

Este notebook guiará a través de la creación de una **red neuronal convolucional (CNN) desde cero** para detectar neumonía en radiografías de tórax.

El objetivo es simple pero poderoso: entrenar un modelo que pueda mirar una radiografía y decidir si muestra un pulmón **NORMAL** o si hay signos de **PNEUMONIA**.

**Dataset utilizado:** Chest X-Ray Images (Pneumonia) desde Kaggle

## 1️⃣ Configuración del Entorno

Aquí preparamos todo lo necesario para que el modelo funcione correctamente.

In [1]:
import importlib.util
import random
import subprocess
import sys
from pathlib import Path
from collections import Counter

def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        package_name = pip_name or import_name
        print(f'Instalando: {package_name}')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])

ensure_package('kagglehub')
ensure_package('sklearn', 'scikit-learn')

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch version: {torch.__version__}')

Device: cuda
PyTorch version: 2.7.0+cu128


## 2️⃣ Descarga del Dataset

Descargamos automáticamente el dataset de radiografías desde Kaggle.

In [2]:
import os
import zipfile

DATASET_HANDLE = 'paultimothymooney/chest-xray-pneumonia'
PROJECT_DIR = Path.cwd()
DOWNLOAD_DIR = PROJECT_DIR / 'data' / 'raw'
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

try:
    downloaded_location = kagglehub.dataset_download(
        DATASET_HANDLE,
        output_dir=str(DOWNLOAD_DIR),
    )
except TypeError:
    downloaded_location = kagglehub.dataset_download(DATASET_HANDLE)

downloaded_path = Path(downloaded_location).expanduser().resolve()

def find_dataset_root(search_root):
    required = {'train', 'val', 'test'}
    visited = []
    search_root = Path(search_root).resolve()
    
    if not search_root.exists():
        return None, visited
    
    if search_root.is_file():
        search_root = search_root.parent
    
    for current, dirnames, _ in os.walk(search_root):
        current_path = Path(current)
        visited.append(current_path)
        children = {dirname.lower(): current_path / dirname for dirname in dirnames}
        
        if required.issubset(children):
            return (current_path, children['train'], children['val'], children['test']), visited
    
    return None, visited

search_roots = [downloaded_path if downloaded_path.is_dir() else downloaded_path.parent, DOWNLOAD_DIR.resolve()]
dataset_structure = None

for search_root in search_roots:
    result, _ = find_dataset_root(search_root)
    if result is not None:
        dataset_structure = result
        break

if dataset_structure is None:
    raise FileNotFoundError('Dataset no encontrado')

DATA_DIR, train_dir, val_dir, test_dir = dataset_structure
print(f'Dataset root: {DATA_DIR}')

Dataset root: C:\Users\Usuario\Desktop\Anaconda\ProyectoFinal\mejorado\data\raw\chest_xray


## 3️⃣ Preparación de Datos

Definimos transformaciones y cargamos los datasets.

In [3]:
IMAGE_SIZE = 128
IMAGE_MEAN = (0.5,)
IMAGE_STD = (0.5,)

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGE_MEAN, IMAGE_STD),
])

eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGE_MEAN, IMAGE_STD),
])

train_dataset_full = datasets.ImageFolder(root=train_dir, transform=train_transform)
train_dataset_eval = datasets.ImageFolder(root=train_dir, transform=eval_transform)
val_dataset = datasets.ImageFolder(root=val_dir, transform=eval_transform)
test_dataset = datasets.ImageFolder(root=test_dir, transform=eval_transform)

class_names = train_dataset_full.classes
num_classes = len(class_names)

train_size = int(0.8 * len(train_dataset_full))
indices = torch.randperm(len(train_dataset_full)).tolist()
train_indices = indices[:train_size]
val_indices = indices[train_size:]

train_subset = Subset(train_dataset_full, train_indices)
val_subset = Subset(train_dataset_eval, val_indices)

BATCH_SIZE = 32
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Clases: {class_names}')
print(f'Train: {len(train_subset)} | Val: {len(val_subset)} | Test: {len(test_dataset)}')

Clases: ['NORMAL', 'PNEUMONIA']
Train: 4172 | Val: 1044 | Test: 624


## 4️⃣ Arquitectura CNN

Construimos una red neuronal convolucional personalizada.

In [4]:
class CNNBaseline(nn.Module):
    def __init__(self, num_classes=2):
        super(CNNBaseline, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(128 * 16 * 16, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x

model = CNNBaseline(num_classes=num_classes)
model = model.to(device)
print('Modelo creado exitosamente')

Modelo creado exitosamente


## 5️⃣ Entrenamiento

Entrenamos el modelo durante 20 épocas.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    return running_loss / total, correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    return running_loss / total, correct / total

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

NUM_EPOCHS = 15
for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f'Epoch {epoch+1}/{NUM_EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}')

print('Entrenamiento completado')

## Gráfico de las epoch


In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, history["train_loss"], label="Train Loss")
plt.plot(epochs, history["val_loss"], label="Validation Loss")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, history["train_acc"], label="Train Accuracy")
plt.plot(epochs, history["val_acc"], label="Validation Accuracy")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

## 6️⃣ Evaluación en Conjunto de Prueba

Evaluamos el modelo con imágenes nuevas.

In [6]:
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f'Test Accuracy: {test_acc:.4f}')

Test Accuracy: 0.7821


## 7️⃣ Clasificación de Imágenes desde `images_sample`

Clasifica imágenes de prueba desde la carpeta `images_sample` en la raíz del proyecto.

In [7]:
def predict_single_image(image_path, model, class_names, device, transform):
    image_path = Path(image_path)
    if not image_path.exists():
        raise FileNotFoundError(f'No existe: {image_path}')
    
    img = Image.open(image_path).convert('L')
    img_tensor = transform(img)
    img_tensor = img_tensor.unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = model(img_tensor)
        probabilities = torch.softmax(outputs, dim=1)
        confidence, predicted_idx = torch.max(probabilities, dim=1)
    
    predicted_class = class_names[predicted_idx.item()]
    confidence_value = confidence.item()
    return predicted_class, confidence_value

SAMPLE_IMAGES_DIR = Path.cwd() / 'images_sample'
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png'}
SAMPLE_IMAGES_DIR.mkdir(exist_ok=True)

sample_images = []
if SAMPLE_IMAGES_DIR.exists():
    for img_file in sorted(SAMPLE_IMAGES_DIR.glob('*')):
        if img_file.suffix.lower() in IMAGE_EXTENSIONS:
            sample_images.append(img_file)

if not sample_images:
    print(f'Carpeta creada: {SAMPLE_IMAGES_DIR}')
    print('Coloca imágenes en esta carpeta y ejecuta de nuevo')
else:
    print(f'Clasificando {len(sample_images)} imágenes:')
    for img_path in sample_images:
        try:
            pred_class, confidence = predict_single_image(img_path, model, class_names, device, eval_transform)
            print(f'{img_path.name}: {pred_class} ({confidence*100:.2f}%)')
        except Exception as e:
            print(f'{img_path.name}: Error - {str(e)}')

Clasificando 10 imágenes:
torax1.jpeg: NORMAL (89.70%)
torax10.jpeg: NORMAL (100.00%)
torax2.jpeg: PNEUMONIA (94.78%)
torax3.jpeg: PNEUMONIA (91.75%)
torax4.jpeg: NORMAL (53.11%)
torax5.jpeg: NORMAL (73.86%)
torax6.jpeg: PNEUMONIA (97.09%)
torax7.jpeg: NORMAL (99.99%)
torax8.jpeg: PNEUMONIA (99.91%)
torax9.jpeg: PNEUMONIA (100.00%)
